# law dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the law_data dataset
law_df = pd.read_csv('/content/law_data.csv')

# Prepare features and target
X = law_df.drop('first_pf', axis=1)
y = law_df['first_pf']

# Sensitive features: race and sex
sensitive_features = {
    'race': law_df['race'],
    'sex': law_df['sex']
}

# Identify numerical and categorical columns
numerical_features = ['LSAT', 'UGPA', 'ZFYA', 'sander_index']
categorical_features = ['region_first']

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Define dataset sizes to test
dataset_sizes = [10000, 1000, 500]

# Models to evaluate
models = {
    'LR': Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(random_state=42))]),
    'RF': Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor), ('classifier', MLPClassifier(random_state=42, max_iter=300))])
}

# Dictionary to store results
results = {'Dataset Size': [], 'Model': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': []}

# Run experiments for each dataset size
for size in dataset_sizes:
    # Stratified sampling to create subset
    sss = StratifiedShuffleSplit(n_splits=1, train_size=min(size, len(X)), random_state=42)
    for train_index, _ in sss.split(X, y):
        X_subset = X.iloc[train_index]
        y_subset = y.iloc[train_index]
        sens_subset_race = sensitive_features['race'].iloc[train_index]
        sens_subset_sex = sensitive_features['sex'].iloc[train_index]

        # Split subset into train and test
        X_train, X_test, y_train, y_test, sens_train_race, sens_test_race, sens_train_sex, sens_test_sex = train_test_split(
            X_subset, y_subset, sens_subset_race, sens_subset_sex,
            test_size=0.2, random_state=42, stratify=y_subset
        )

        # Evaluate each model
        for name, model in models.items():
            # Fit and predict
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            # Calculate accuracy
            acc = accuracy_score(y_test, y_pred)

            # Fairness metrics for each sensitive feature
            for sens_name, sens_test in [('race', sens_test_race), ('sex', sens_test_sex)]:
                dp_diff = fm.demographic_parity_difference(y_test, y_pred, sensitive_features=sens_test)
                eo_diff = fm.equalized_odds_difference(y_test, y_pred, sensitive_features=sens_test)

                # Store results
                results['Dataset Size'].append(size)
                results['Model'].append(name)
                results['Sensitive'].append(sens_name)
                results['DP Diff'].append(round(dp_diff, 4))
                results['EO Diff'].append(round(eo_diff, 4))
                results['Accuracy'].append(round(acc, 4))

# Display results as DataFrame
results_df = pd.DataFrame(results)
print("Law Dataset Fairness Results for Different Sizes:")
print(results_df)

Law Dataset Fairness Results for Different Sizes:
    Dataset Size Model Sensitive  DP Diff  EO Diff  Accuracy
0          10000    LR      race   0.2168   0.4850    0.8990
1          10000    LR       sex   0.0211   0.0598    0.8990
2          10000    RF      race   0.1857   0.3578    0.8815
3          10000    RF       sex   0.0207   0.0161    0.8815
4          10000   MLP      race   0.2023   0.4109    0.8950
5          10000   MLP       sex   0.0216   0.0343    0.8950
6           1000    LR      race   0.2157   0.5536    0.9000
7           1000    LR       sex   0.0010   0.0855    0.9000
8           1000    RF      race   0.1608   0.3036    0.9000
9           1000    RF       sex   0.0006   0.1197    0.9000
10          1000   MLP      race   0.2255   0.6071    0.8950
11          1000   MLP       sex   0.0085   0.0513    0.8950
12           500    LR      race   0.3524   0.6333    0.9400
13           500    LR       sex   0.0282   0.2917    0.9400
14           500    RF      race   

In [ ]:
"""
Finetuning for Law dataset with fairness evaluation across different dataset sizes.
Subsamples data at specified levels (10000, 1000, 500) instead of using full dataset.
"""
from functools import partial
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from torch.optim import Adam, Optimizer
from torch.utils.data import DataLoader
from tqdm import tqdm
from tabpfn import TabPFNClassifier
from tabpfn.finetune_utils import clone_model_for_evaluation
from tabpfn.utils import meta_dataset_collator
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

def prepare_data(config: dict, dataset_size: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Loads and splits the Law dataset with subsampling."""
    print(f"--- 1. Data Preparation (Size: {dataset_size}) ---")
    law_df = pd.read_csv('/content/law_data.csv')
    X_all = law_df.drop('first_pf', axis=1)
    y_all = law_df['first_pf']
    race_all = law_df['race']
    sex_all = law_df['sex']

    # Subsample data
    sss = StratifiedShuffleSplit(n_splits=1, train_size=min(dataset_size, len(X_all)), random_state=config["random_seed"])
    for train_index, _ in sss.split(X_all, y_all):
        X_subset = X_all.iloc[train_index]
        y_subset = y_all.iloc[train_index]
        race_subset = race_all.iloc[train_index]
        sex_subset = sex_all.iloc[train_index]

    # Preprocessing: Standardize numerical, one-hot categorical
    numerical_features = X_subset.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X_subset.select_dtypes(include=['object']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
        ])
    X_subset_processed = preprocessor.fit_transform(X_subset)

    # Split subset
    splitter = partial(
        train_test_split,
        test_size=config["valid_set_ratio"],
        random_state=config["random_seed"],
    )
    X_train, X_test, y_train, y_test = splitter(X_subset_processed, y_subset, stratify=y_subset)
    _, race_test, _, _ = splitter(race_subset, y_subset, stratify=y_subset)
    _, sex_test, _, _ = splitter(sex_subset, y_subset, stratify=y_subset)

    print(
        f"Loaded subset (size: {dataset_size}) and split: {X_train.shape[0]} train, {X_test.shape[0]} test samples."
    )
    print("---------------------------\n")
    return X_train, X_test, y_train, y_test, race_test, sex_test

def setup_model_and_optimizer(config: dict) -> tuple[TabPFNClassifier, Optimizer, dict]:
    """Initializes the TabPFN classifier, optimizer, and training configs."""
    print("--- 2. Model and Optimizer Setup ---")
    classifier_config = {
        "ignore_pretraining_limits": True,
        "device": config["device"],
        "n_estimators": 2,
        "random_state": config["random_seed"],
        "inference_precision": torch.float32,
    }
    classifier = TabPFNClassifier(
        **classifier_config, fit_mode="batched", differentiable_input=False
    )
    classifier._initialize_model_variables()
    # Optimizer uses finetuning-specific learning rate
    optimizer = Adam(
        classifier.model_.parameters(), lr=config["finetuning"]["learning_rate"]
    )
    print(f"Using device: {config['device']}")
    print(f"Optimizer: Adam, Finetuning LR: {config['finetuning']['learning_rate']}")
    print("----------------------------------\n")
    return classifier, optimizer, classifier_config

def evaluate_model(
    classifier: TabPFNClassifier,
    eval_config: dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    race_test: np.ndarray,
    sex_test: np.ndarray,
) -> tuple[float, float, float, pd.DataFrame]:
    """Evaluates utility and fairness on the test set."""
    eval_classifier = clone_model_for_evaluation(
        classifier, eval_config, TabPFNClassifier
    )
    eval_classifier.fit(X_train, y_train)
    try:
        probabilities = eval_classifier.predict_proba(X_test)
        predictions = (probabilities[:, 1] > 0.5).astype(int)
        roc_auc = roc_auc_score(y_test, probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)
        log_loss_score = log_loss(y_test, probabilities)
        # Fairness metrics
        sensitive = {'race': race_test, 'sex': sex_test}
        results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
        for sens_name, sens_test in sensitive.items():
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
        fairness_df = pd.DataFrame(results)
    except Exception as e:
        print(f"An error occurred during evaluation: {e}")
        roc_auc, accuracy, log_loss_score = np.nan, np.nan, np.nan
        fairness_df = pd.DataFrame()
    return roc_auc, accuracy, log_loss_score, fairness_df

def main() -> None:
    """Main function to configure and run the finetuning workflow with fairness eval for different dataset sizes."""
    # --- Master Configuration ---
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "valid_set_ratio": 0.3,
        "n_inference_context_samples": 5000,  # Reduced for full data to save memory
    }
    config["finetuning"] = {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 5000,  # Adjusted for subsampled data
    }

    # Define dataset sizes to test
    dataset_sizes = [10000, 1000, 500]

    # Loop over dataset sizes
    for dataset_size in dataset_sizes:
        print(f"\n=== Processing Dataset Size: {dataset_size} ===\n")
        # --- Setup Data, Model, and Dataloader ---
        X_train, X_test, y_train, y_test, race_test, sex_test = prepare_data(config, dataset_size)
        classifier, optimizer, classifier_config = setup_model_and_optimizer(config)
        splitter = partial(train_test_split, test_size=config["valid_set_ratio"])
        training_datasets = classifier.get_preprocessed_datasets(
            X_train, y_train, splitter, config["finetuning"]["batch_size"]
        )
        finetuning_dataloader = DataLoader(
            training_datasets,
            batch_size=config["finetuning"]["meta_batch_size"],
            collate_fn=meta_dataset_collator,
        )
        loss_function = torch.nn.CrossEntropyLoss()
        eval_config = {
            **classifier_config,
            "inference_config": {
                "SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]
            },
        }

        # --- Finetuning and Evaluation Loop ---
        print("--- 3. Starting Finetuning & Evaluation ---")
        for epoch in range(config["finetuning"]["epochs"] + 1):
            if epoch > 0:
                # Finetuning Step
                progress_bar = tqdm(finetuning_dataloader, desc=f"Finetuning Epoch {epoch} (Size: {dataset_size})")
                for (
                    X_train_batch,
                    X_test_batch,
                    y_train_batch,
                    y_test_batch,
                    cat_ixs,
                    confs,
                ) in progress_bar:
                    if len(np.unique(y_train_batch)) != len(np.unique(y_test_batch)):
                        continue
                    optimizer.zero_grad()
                    classifier.fit_from_preprocessed(
                        X_train_batch, y_train_batch, cat_ixs, confs
                    )
                    predictions = classifier.forward(X_test_batch, return_logits=True)
                    loss = loss_function(predictions, y_test_batch.to(config["device"]))
                    loss.backward()
                    optimizer.step()
                    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

            # Evaluation Step
            epoch_roc, epoch_acc, epoch_log_loss, fairness_df = evaluate_model(
                classifier, eval_config, X_train, y_train, X_test, y_test, race_test, sex_test
            )
            status = "Initial" if epoch == 0 else f"Epoch {epoch}"
            print(
                f"📊 {status} Utility (Size: {dataset_size}) | Test ROC: {epoch_roc:.4f}, Accuracy: {epoch_acc:.4f}, Log Loss: {epoch_log_loss:.4f}\n"
            )
            print(f"{status} Fairness Results (Size: {dataset_size}):\n{fairness_df}\n")

        print(f"--- ✅ Finetuning Finished for Size {dataset_size} ---")

if __name__ == "__main__":
    main()


=== Processing Dataset Size: 10000 ===

--- 1. Data Preparation (Size: 10000) ---
Loaded subset (size: 10000) and split: 7000 train, 3000 test samples.
---------------------------

--- 2. Model and Optimizer Setup ---
Using device: cuda
Optimizer: Adam, Finetuning LR: 1e-05
----------------------------------

--- 3. Starting Finetuning & Evaluation ---
📊 Initial Utility (Size: 10000) | Test ROC: 0.8294, Accuracy: 0.8980, Log Loss: 0.2765

Initial Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.1731   0.4059
1       sex   0.0125   0.0093



Finetuning Epoch 1 (Size: 10000): 100%|██████████| 2/2 [00:02<00:00,  1.27s/it, loss=0.2555]


📊 Epoch 1 Utility (Size: 10000) | Test ROC: 0.8296, Accuracy: 0.8980, Log Loss: 0.2759

Epoch 1 Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.2044   0.4534
1       sex   0.0147   0.0099



Finetuning Epoch 2 (Size: 10000): 100%|██████████| 2/2 [00:02<00:00,  1.26s/it, loss=0.2453]


📊 Epoch 2 Utility (Size: 10000) | Test ROC: 0.8294, Accuracy: 0.8967, Log Loss: 0.2753

Epoch 2 Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.2078   0.4534
1       sex   0.0138   0.0089



...

📊 Epoch 8 Utility (Size: 500) | Test ROC: 0.8987, Accuracy: 0.9267, Log Loss: 0.2229

Epoch 8 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.3601   0.6389
1       sex   0.0339   0.1364



Finetuning Epoch 9 (Size: 500): 100%|██████████| 1/1 [00:00<00:00,  3.23it/s, loss=0.2810]


📊 Epoch 9 Utility (Size: 500) | Test ROC: 0.8983, Accuracy: 0.9200, Log Loss: 0.2233

Epoch 9 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.4226   0.6389
1       sex   0.0214   0.1364



Finetuning Epoch 10 (Size: 500): 100%|██████████| 1/1 [00:00<00:00,  3.20it/s, loss=0.2720]


📊 Epoch 10 Utility (Size: 500) | Test ROC: 0.8978, Accuracy: 0.9200, Log Loss: 0.2237

Epoch 10 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.4226   0.6389
1       sex   0.0214   0.1364

--- ✅ Finetuning Finished for Size 500 ---


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from tabpfn import TabPFNClassifier
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the law_data dataset
law_df = pd.read_csv('/content/law_data.csv')

# Prepare features and target
X = law_df.drop('first_pf', axis=1)
y = law_df['first_pf']

# Sensitive features: race and sex
sensitive_features = {
    'race': law_df['race'],
    'sex': law_df['sex']
}

# Identify numerical and categorical columns
numerical_features = ['LSAT', 'UGPA', 'ZFYA', 'sander_index', 'race', 'sex']
categorical_features = ['region_first']

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ])

# Define dataset sizes to test
dataset_sizes = [10000, 1000, 500]

# Dictionary to store results
results = {'Dataset Size': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': [], 'ROC AUC': []}

# Run experiments for each dataset size
for size in dataset_sizes:
    # Stratified sampling to create subset
    sss = StratifiedShuffleSplit(n_splits=1, train_size=min(size, len(X)), random_state=42)
    for train_index, _ in sss.split(X, y):
        X_subset = X.iloc[train_index]
        y_subset = y.iloc[train_index]
        sens_subset_race = sensitive_features['race'].iloc[train_index]
        sens_subset_sex = sensitive_features['sex'].iloc[train_index]

        # Apply preprocessing
        X_subset_processed = preprocessor.fit_transform(X_subset)

        # Split subset into train and test
        X_train, X_test, y_train, y_test, sens_train_race, sens_test_race, sens_train_sex, sens_test_sex = train_test_split(
            X_subset_processed, y_subset, sens_subset_race, sens_subset_sex,
            test_size=0.5, random_state=42, stratify=y_subset
        )

        # Initialize and fit the classifier
        clf = TabPFNClassifier(ignore_pretraining_limits=True)
        clf.fit(X_train, y_train)

        # Predict probabilities and labels
        prediction_probabilities = clf.predict_proba(X_test)
        predictions = clf.predict(X_test)

        # Calculate performance metrics
        roc_auc = roc_auc_score(y_test, prediction_probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)

        # Fairness metrics
        for sens_name, sens_test in [('race', sens_test_race), ('sex', sens_test_sex)]:
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)

            # Store results
            results['Dataset Size'].append(size)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
            results['Accuracy'].append(round(accuracy, 4))
            results['ROC AUC'].append(round(roc_auc, 4))

# Display results as DataFrame
results_df = pd.DataFrame(results)
print("Law Dataset TabPFN Fairness Results for Different Sizes:")
print(results_df)

Law Dataset TabPFN Fairness Results for Different Sizes:
   Dataset Size Sensitive  DP Diff  EO Diff  Accuracy  ROC AUC
0         10000      race   0.1814   0.3379    0.8966   0.8362
1         10000       sex   0.0133   0.0127    0.8966   0.8362
2          1000      race   0.2560   0.5625    0.9040   0.8692
3          1000       sex   0.0191   0.2271    0.9040   0.8692
4           500      race   0.0542   0.0872    0.9000   0.8756
5           500       sex   0.0088   0.1696    0.9000   0.8756


In [ ]:
# law full dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from tabpfn import TabPFNClassifier
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the law_data dataset
law_df = pd.read_csv('/content/law_data.csv')

# Prepare features and target
X = law_df.drop('first_pf', axis=1)
y = law_df['first_pf']

# Sensitive features: race and sex
sensitive_features = {
    'race': law_df['race'],
    'sex': law_df['sex']
}

# Identify numerical and categorical columns
numerical_features = ['LSAT', 'UGPA', 'ZFYA', 'sander_index', 'race', 'sex']  # Include race/sex as numerical if encoded
categorical_features = ['region_first']

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ])

# Apply preprocessing
X_processed = preprocessor.fit_transform(X)

# Split data (test_size=0.5)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.5, random_state=42, stratify=y)

# Split sensitive features accordingly
sens_splits = {}
for sens_name, sens in sensitive_features.items():
    _, sens_test, _, _ = train_test_split(sens, y, test_size=0.5, random_state=42, stratify=y)
    sens_splits[sens_name] = sens_test

# Initialize and fit the classifier
clf = TabPFNClassifier(ignore_pretraining_limits=True)
clf.fit(X_train, y_train)

# Predict probabilities and labels
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Calculate performance metrics
roc_auc = roc_auc_score(y_test, prediction_probabilities[:, 1])
accuracy = accuracy_score(y_test, predictions)

print("Law Data Dataset - ROC AUC:", round(roc_auc, 4))
print("Law Data Dataset - Accuracy:", round(accuracy, 4))

# Fairness metrics
results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
for sens_name, sens_test in sens_splits.items():
    dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
    eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)

    results['Sensitive'].append(sens_name)
    results['DP Diff'].append(round(dp_diff, 4))
    results['EO Diff'].append(round(eo_diff, 4))

# Display fairness results
law_fairness_df = pd.DataFrame(results)
print("\nLaw Data Dataset TabPFN Fairness Results:")
print(law_fairness_df)

Law Data Dataset - ROC AUC: 0.8574
Law Data Dataset - Accuracy: 0.9001

Law Data Dataset TabPFN Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1942   0.4167
1       sex   0.0202   0.0663


In [ ]:
"""
Full dataset fine-tuning for Law dataset with fairness evaluation.
No subsampling; uses entire dataset.
"""
from functools import partial
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from torch.optim import Adam, Optimizer
from torch.utils.data import DataLoader
from tqdm import tqdm
from tabpfn import TabPFNClassifier
from tabpfn.finetune_utils import clone_model_for_evaluation
from tabpfn.utils import meta_dataset_collator
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

def prepare_data(config: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Loads and splits the full Law dataset, including sensitive features."""
    print("--- 1. Data Preparation ---")
    law_df = pd.read_csv('/content/law_data.csv')
    X_all = law_df.drop('first_pf', axis=1)
    y_all = law_df['first_pf']
    race_all = law_df['race']
    sex_all = law_df['sex']

    # Preprocessing: Standardize numerical, one-hot categorical
    numerical_features = X_all.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X_all.select_dtypes(include=['object']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
        ])
    X_all_processed = preprocessor.fit_transform(X_all)

    # No subsampling: use full data
    X_all = X_all_processed
    y = y_all.values
    race = race_all.values
    sex = sex_all.values

    splitter = partial(
        train_test_split,
        test_size=config["valid_set_ratio"],
        random_state=config["random_seed"],
    )
    X_train, X_test, y_train, y_test = splitter(X_all, y, stratify=y)
    _, race_test, _, _ = splitter(race, y, stratify=y)
    _, sex_test, _, _ = splitter(sex, y, stratify=y)

    print(
        f"Loaded full data and split: {X_train.shape[0]} train, {X_test.shape[0]} test samples."
    )
    print("---------------------------\n")
    return X_train, X_test, y_train, y_test, race_test, sex_test

def setup_model_and_optimizer(config: dict) -> tuple[TabPFNClassifier, Optimizer, dict]:
    """Initializes the TabPFN classifier, optimizer, and training configs."""
    print("--- 2. Model and Optimizer Setup ---")
    classifier_config = {
        "ignore_pretraining_limits": True,
        "device": config["device"],
        "n_estimators": 2,
        "random_state": config["random_seed"],
        "inference_precision": torch.float32,
    }
    classifier = TabPFNClassifier(
        **classifier_config, fit_mode="batched", differentiable_input=False
    )
    classifier._initialize_model_variables()
    # Optimizer uses finetuning-specific learning rate
    optimizer = Adam(
        classifier.model_.parameters(), lr=config["finetuning"]["learning_rate"]
    )
    print(f"Using device: {config['device']}")
    print(f"Optimizer: Adam, Finetuning LR: {config['finetuning']['learning_rate']}")
    print("----------------------------------\n")
    return classifier, optimizer, classifier_config

def evaluate_model(
    classifier: TabPFNClassifier,
    eval_config: dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    race_test: np.ndarray,
    sex_test: np.ndarray,
) -> tuple[float, float, float, pd.DataFrame]:
    """Evaluates utility and fairness on the test set."""
    eval_classifier = clone_model_for_evaluation(
        classifier, eval_config, TabPFNClassifier
    )
    eval_classifier.fit(X_train, y_train)
    try:
        probabilities = eval_classifier.predict_proba(X_test)
        predictions = (probabilities[:, 1] > 0.5).astype(int)
        roc_auc = roc_auc_score(y_test, probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)
        log_loss_score = log_loss(y_test, probabilities)

        # Fairness metrics
        sensitive = {'race': race_test, 'sex': sex_test}
        results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
        for sens_name, sens_test in sensitive.items():
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
        fairness_df = pd.DataFrame(results)
    except Exception as e:
        print(f"An error occurred during evaluation: {e}")
        roc_auc, accuracy, log_loss_score = np.nan, np.nan, np.nan
        fairness_df = pd.DataFrame()
    return roc_auc, accuracy, log_loss_score, fairness_df

def main() -> None:
    """Main function to configure and run the finetuning workflow with fairness eval."""
    # --- Master Configuration ---
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "valid_set_ratio": 0.3,
        "n_inference_context_samples": 5000,  # Reduced for full data to save memory
    }
    config["finetuning"] = {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 5000,  # Adjusted for full data
    }
    # --- Setup Data, Model, and Dataloader ---
    X_train, X_test, y_train, y_test, race_test, sex_test = prepare_data(config)
    classifier, optimizer, classifier_config = setup_model_and_optimizer(config)
    splitter = partial(train_test_split, test_size=config["valid_set_ratio"])
    training_datasets = classifier.get_preprocessed_datasets(
        X_train, y_train, splitter, config["finetuning"]["batch_size"]
    )
    finetuning_dataloader = DataLoader(
        training_datasets,
        batch_size=config["finetuning"]["meta_batch_size"],
        collate_fn=meta_dataset_collator,
    )
    loss_function = torch.nn.CrossEntropyLoss()
    eval_config = {
        **classifier_config,
        "inference_config": {
            "SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]
        },
    }
    # --- Finetuning and Evaluation Loop ---
    print("--- 3. Starting Finetuning & Evaluation ---")
    for epoch in range(config["finetuning"]["epochs"] + 1):
        if epoch > 0:
            # Finetuning Step
            progress_bar = tqdm(finetuning_dataloader, desc=f"Finetuning Epoch {epoch}")
            for (
                X_train_batch,
                X_test_batch,
                y_train_batch,
                y_test_batch,
                cat_ixs,
                confs,
            ) in progress_bar:
                if len(np.unique(y_train_batch)) != len(np.unique(y_test_batch)):
                    continue
                optimizer.zero_grad()
                classifier.fit_from_preprocessed(
                    X_train_batch, y_train_batch, cat_ixs, confs
                )
                predictions = classifier.forward(X_test_batch, return_logits=True)
                loss = loss_function(predictions, y_test_batch.to(config["device"]))
                loss.backward()
                optimizer.step()
                progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        # Evaluation Step
        epoch_roc, epoch_acc, epoch_log_loss, fairness_df = evaluate_model(
            classifier, eval_config, X_train, y_train, X_test, y_test, race_test, sex_test
        )
        status = "Initial" if epoch == 0 else f"Epoch {epoch}"
        print(
            f"📊 {status} Utility | Test ROC: {epoch_roc:.4f}, Accuracy: {epoch_acc:.4f}, Log Loss: {epoch_log_loss:.4f}\n"
        )
        print(f"{status} Fairness Results:\n{fairness_df}\n")
    print("--- ✅ Finetuning Finished ---")

if __name__ == "__main__":
    main()

--- 1. Data Preparation ---
Loaded full data and split: 15253 train, 6538 test samples.
---------------------------

--- 2. Model and Optimizer Setup ---
Using device: cuda
Optimizer: Adam, Finetuning LR: 1e-05
----------------------------------

--- 3. Starting Finetuning & Evaluation ---
📊 Initial Utility | Test ROC: 0.8538, Accuracy: 0.8998, Log Loss: 0.2584

Initial Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1698   0.3803
1       sex   0.0252   0.1223



Finetuning Epoch 1: 100%|██████████| 4/4 [00:05<00:00,  1.41s/it, loss=0.2790]


📊 Epoch 1 Utility | Test ROC: 0.8544, Accuracy: 0.9004, Log Loss: 0.2553

Epoch 1 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1973   0.4174
1       sex   0.0269   0.1162



Finetuning Epoch 2: 100%|██████████| 4/4 [00:05<00:00,  1.41s/it, loss=0.2447]


📊 Epoch 2 Utility | Test ROC: 0.8549, Accuracy: 0.8995, Log Loss: 0.2556

Epoch 2 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.2085   0.4187
1       sex   0.0264   0.1133



...

📊 Epoch 8 Utility | Test ROC: 0.8547, Accuracy: 0.8998, Log Loss: 0.2554

Epoch 8 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1656   0.3721
1       sex   0.0230   0.1114



Finetuning Epoch 9: 100%|██████████| 4/4 [00:05<00:00,  1.41s/it, loss=0.2308]


📊 Epoch 9 Utility | Test ROC: 0.8547, Accuracy: 0.9006, Log Loss: 0.2556

Epoch 9 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1739   0.3815
1       sex   0.0234   0.1029



Finetuning Epoch 10: 100%|██████████| 4/4 [00:05<00:00,  1.41s/it, loss=0.2618]


📊 Epoch 10 Utility | Test ROC: 0.8548, Accuracy: 0.9010, Log Loss: 0.2556

Epoch 10 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1887   0.4081
1       sex   0.0238   0.1109

--- ✅ Finetuning Finished ---


In [ ]:
# Import necessary libraries (reuse if running after previous code)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the law_data dataset
law_df = pd.read_csv('/content/law_data.csv')

# Prepare features and target
X = law_df.drop('first_pf', axis=1)
y = law_df['first_pf']

# Sensitive features: race and sex
sensitive_features = {
    'race': law_df['race'],
    'sex': law_df['sex']
}

# Identify numerical and categorical columns
numerical_features = ['LSAT', 'UGPA', 'ZFYA', 'sander_index']  # Based on data info
categorical_features = ['region_first']

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train models
models = {
    'LR': Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(random_state=42))]),
    'RF': Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor), ('classifier', MLPClassifier(random_state=42, max_iter=300))])
}

# Dictionary to store results
results = {'Model': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': []}

for name, model in models.items():
    # Fit and predict
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    # Fairness metrics for each sensitive feature
    for sens_name, sens in sensitive_features.items():
        sens_train, sens_test = train_test_split(sens, test_size=0.2, random_state=42, stratify=y)
        dp_diff = fm.demographic_parity_difference(y_test, y_pred, sensitive_features=sens_test)
        eo_diff = fm.equalized_odds_difference(y_test, y_pred, sensitive_features=sens_test)

        results['Model'].append(name)
        results['Sensitive'].append(sens_name)
        results['DP Diff'].append(round(dp_diff, 4))
        results['EO Diff'].append(round(eo_diff, 4))
        results['Accuracy'].append(round(acc, 4))

# Display results as DataFrame
law_results_df = pd.DataFrame(results)
print("Law Data Fairness Results:")
print(law_results_df)

Law Data Fairness Results:
  Model Sensitive  DP Diff  EO Diff  Accuracy
0    LR      race   0.2029   0.4098    0.8997
1    LR       sex   0.0213   0.1111    0.8997
2    RF      race   0.1963   0.3571    0.8880
3    RF       sex   0.0270   0.1117    0.8880
4   MLP      race   0.1905   0.3887    0.8981
5   MLP       sex   0.0220   0.0902    0.8981


# bank dataset

In [ ]:
# ============================================
# 🏦 Bank Personal Loan Dataset - TabPFN Fairness Analysis
# ============================================

# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from tabpfn import TabPFNClassifier
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# ============================================
# Load the bank dataset (Excel with sheets)
# ============================================
bank_df = pd.read_excel('/content/Bank_Personal_Loan_Modelling.xlsx', sheet_name='Data')

# ============================================
# Prepare features and target
# ============================================
# Target: Personal Loan (0 = No, 1 = Yes)
X = bank_df.drop(['Personal Loan', 'ID', 'ZIP Code'], axis=1)
y = bank_df['Personal Loan']

# Sensitive features: Education, Family
sensitive_features = {
    'Education': bank_df['Education'],
    'Family': bank_df['Family']
}

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# ============================================
# Preprocessing
# ============================================
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ])

# Apply preprocessing
X_processed = preprocessor.fit_transform(X)

# ============================================
# Split data (test_size=0.5 for TabPFN)
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.5, random_state=42, stratify=y)

# Split sensitive features accordingly
sens_splits = {}
for sens_name, sens in sensitive_features.items():
    _, sens_test, _, _ = train_test_split(sens, y, test_size=0.5, random_state=42, stratify=y)
    sens_splits[sens_name] = sens_test

# ============================================
# Initialize and fit the TabPFN classifier
# ============================================
clf = TabPFNClassifier(ignore_pretraining_limits=True)
clf.fit(X_train, y_train)

# ============================================
# Predict probabilities and labels
# ============================================
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# ============================================
# Performance metrics
# ============================================
roc_auc = roc_auc_score(y_test, prediction_probabilities[:, 1])
accuracy = accuracy_score(y_test, predictions)

print("Bank Dataset - ROC AUC:", round(roc_auc, 4))
print("Bank Dataset - Accuracy:", round(accuracy, 4))

# ============================================
# Fairness metrics
# ============================================
results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
for sens_name, sens_test in sens_splits.items():
    dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
    eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)

    results['Sensitive'].append(sens_name)
    results['DP Diff'].append(round(dp_diff, 4))
    results['EO Diff'].append(round(eo_diff, 4))

# ============================================
# Display fairness results
# ============================================
bank_fairness_df = pd.DataFrame(results)
print("\nBank Dataset TabPFN Fairness Results:")
print(bank_fairness_df)


tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

Bank Dataset - ROC AUC: 0.9979
Bank Dataset - Accuracy: 0.9876

Bank Dataset TabPFN Fairness Results:
   Sensitive  DP Diff  EO Diff
0  Education   0.1024   0.0885
1     Family   0.0614   0.1430


In [ ]:
# --- Bank Personal Loan Dataset ---
# Load the dataset
bank_df = pd.read_excel('/content/Bank_Personal_Loan_Modelling.xlsx', sheet_name='Data')

# Check for missing values
print("\nBank Dataset Missing Values:")
print(bank_df.isnull().sum())

# Drop unnecessary or ID-like columns
bank_df = bank_df.drop(['ID', 'ZIP Code'], axis=1)

# Prepare features (X) and target (y)
X_bank = bank_df.drop('Personal Loan', axis=1)
y_bank = bank_df['Personal Loan']

# Sensitive features: Education, Family
sensitive_features_bank = {
    'Education': bank_df['Education'],
    'Family': bank_df['Family']
}

# Identify numerical and categorical columns
numerical_features_bank = X_bank.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_bank = X_bank.select_dtypes(include=['object']).columns.tolist()

# ===============================
# 🧩 Preprocessing and Modeling
# ===============================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import fairlearn.metrics as fm
import pandas as pd

# Preprocessing pipeline
preprocessor_bank = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features_bank),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_bank)
    ])

# Split data
X_train_bank, X_test_bank, y_train_bank, y_test_bank = train_test_split(
    X_bank, y_bank, test_size=0.2, random_state=42, stratify=y_bank)

# Define models
models_bank = {
    'LR': Pipeline([('preprocessor', preprocessor_bank), ('classifier', LogisticRegression(random_state=42))]),
    'RF': Pipeline([('preprocessor', preprocessor_bank), ('classifier', RandomForestClassifier(random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor_bank), ('classifier', MLPClassifier(random_state=42, max_iter=300))])
}

# ===============================
# ⚖️ Fairness Evaluation
# ===============================
results_bank = {'Model': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': []}

for name, model in models_bank.items():
    model.fit(X_train_bank, y_train_bank)
    y_pred_bank = model.predict(X_test_bank)
    acc = accuracy_score(y_test_bank, y_pred_bank)

    # Fairness metrics for each sensitive feature
    for sens_name, sens in sensitive_features_bank.items():
        sens_train, sens_test = train_test_split(sens, test_size=0.2, random_state=42, stratify=y_bank)
        dp_diff = fm.demographic_parity_difference(y_test_bank, y_pred_bank, sensitive_features=sens_test)
        eo_diff = fm.equalized_odds_difference(y_test_bank, y_pred_bank, sensitive_features=sens_test)

        results_bank['Model'].append(name)
        results_bank['Sensitive'].append(sens_name)
        results_bank['DP Diff'].append(round(dp_diff, 4))
        results_bank['EO Diff'].append(round(eo_diff, 4))
        results_bank['Accuracy'].append(round(acc, 4))

# Display results
bank_results_df = pd.DataFrame(results_bank)
print("\nBank Dataset Fairness Results:")
print(bank_results_df)



Bank Dataset Missing Values:
ID                    0
Age                   0
Experience            0
Income                0
ZIP Code              0
Family                0
CCAvg                 0
Education             0
Mortgage              0
Personal Loan         0
Securities Account    0
CD Account            0
Online                0
CreditCard            0
dtype: int64

Bank Dataset Fairness Results:
  Model  Sensitive  DP Diff  EO Diff  Accuracy
0    LR  Education   0.1031   0.5794     0.955
1    LR     Family   0.0464   0.1162     0.955
2    RF  Education   0.1041   0.1429     0.992
3    RF     Family   0.0821   0.0938     0.992
4   MLP  Education   0.1212   0.1667     0.988
5   MLP     Family   0.0709   0.1250     0.988


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
"""
Full dataset fine-tuning for Bank dataset with fairness evaluation.
"""
from functools import partial
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from torch.optim import Adam, Optimizer
from torch.utils.data import DataLoader
from tqdm import tqdm
from tabpfn import TabPFNClassifier
from tabpfn.finetune_utils import clone_model_for_evaluation
from tabpfn.utils import meta_dataset_collator
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# 1. Data Preparation
# ============================================================
def prepare_data(config: dict):
    """Loads and splits the full Bank dataset, including sensitive features."""
    print("--- 1. Data Preparation ---")
    bank_df = pd.read_excel('/content/Bank_Personal_Loan_Modelling.xlsx', sheet_name='Data')

    # Drop irrelevant identifiers
    if 'ID' in bank_df.columns:
        bank_df = bank_df.drop(columns=['ID'])
    if 'ZIP Code' in bank_df.columns:
        bank_df = bank_df.drop(columns=['ZIP Code'])

    # Target
    X_all = bank_df.drop('Personal Loan', axis=1)
    y_all = bank_df['Personal Loan']

    # Sensitive features: Education, Family
    edu_all = bank_df['Education']
    fam_all = bank_df['Family']

    # Preprocessing
    numerical_features = X_all.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X_all.select_dtypes(include=['object']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
        ])
    X_all_processed = preprocessor.fit_transform(X_all)

    # Use full dataset, split train/test
    X_all = X_all_processed
    y = y_all.values
    edu = edu_all.values
    fam = fam_all.values

    splitter = partial(
        train_test_split,
        test_size=config["valid_set_ratio"],
        random_state=config["random_seed"],
    )
    X_train, X_test, y_train, y_test = splitter(X_all, y, stratify=y)
    _, edu_test, _, _ = splitter(edu, y, stratify=y)
    _, fam_test, _, _ = splitter(fam, y, stratify=y)

    print(f"Loaded full data and split: {X_train.shape[0]} train, {X_test.shape[0]} test samples.")
    print("---------------------------\n")

    return X_train, X_test, y_train, y_test, edu_test, fam_test


# ============================================================
# 2. Model and Optimizer Setup
# ============================================================
def setup_model_and_optimizer(config: dict):
    """Initializes the TabPFN classifier and optimizer."""
    print("--- 2. Model and Optimizer Setup ---")
    classifier_config = {
        "ignore_pretraining_limits": True,
        "device": config["device"],
        "n_estimators": 2,
        "random_state": config["random_seed"],
        "inference_precision": torch.float32,
    }
    classifier = TabPFNClassifier(
        **classifier_config, fit_mode="batched", differentiable_input=False
    )
    classifier._initialize_model_variables()
    optimizer = Adam(
        classifier.model_.parameters(), lr=config["finetuning"]["learning_rate"]
    )
    print(f"Using device: {config['device']}")
    print(f"Optimizer: Adam, Finetuning LR: {config['finetuning']['learning_rate']}")
    print("----------------------------------\n")
    return classifier, optimizer, classifier_config


# ============================================================
# 3. Evaluation (Utility + Fairness)
# ============================================================
def evaluate_model(
    classifier: TabPFNClassifier,
    eval_config: dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    edu_test: np.ndarray,
    fam_test: np.ndarray,
):
    """Evaluates performance and fairness on test set."""
    eval_classifier = clone_model_for_evaluation(classifier, eval_config, TabPFNClassifier)
    eval_classifier.fit(X_train, y_train)

    try:
        probabilities = eval_classifier.predict_proba(X_test)
        predictions = (probabilities[:, 1] > 0.5).astype(int)
        roc_auc = roc_auc_score(y_test, probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)
        log_loss_score = log_loss(y_test, probabilities)

        # Fairness metrics
        sensitive = {'Education': edu_test, 'Family': fam_test}
        results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
        for sens_name, sens_test in sensitive.items():
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
        fairness_df = pd.DataFrame(results)
    except Exception as e:
        print(f"An error occurred during evaluation: {e}")
        roc_auc, accuracy, log_loss_score = np.nan, np.nan, np.nan
        fairness_df = pd.DataFrame()

    return roc_auc, accuracy, log_loss_score, fairness_df


# ============================================================
# 4. Main Routine
# ============================================================
def main():
    """Main function to configure and run the finetuning workflow with fairness eval."""
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "valid_set_ratio": 0.3,
        "n_inference_context_samples": 5000,
    }
    config["finetuning"] = {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 5000,
    }

    # Setup data and model
    X_train, X_test, y_train, y_test, edu_test, fam_test = prepare_data(config)
    classifier, optimizer, classifier_config = setup_model_and_optimizer(config)

    splitter = partial(train_test_split, test_size=config["valid_set_ratio"])
    training_datasets = classifier.get_preprocessed_datasets(
        X_train, y_train, splitter, config["finetuning"]["batch_size"]
    )
    finetuning_dataloader = DataLoader(
        training_datasets,
        batch_size=config["finetuning"]["meta_batch_size"],
        collate_fn=meta_dataset_collator,
    )

    loss_function = torch.nn.CrossEntropyLoss()
    eval_config = {
        **classifier_config,
        "inference_config": {"SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]},
    }

    # Finetuning loop
    print("--- 3. Starting Finetuning & Evaluation ---")
    for epoch in range(config["finetuning"]["epochs"] + 1):
        if epoch > 0:
            progress_bar = tqdm(finetuning_dataloader, desc=f"Finetuning Epoch {epoch}")
            for (
                X_train_batch,
                X_test_batch,
                y_train_batch,
                y_test_batch,
                cat_ixs,
                confs,
            ) in progress_bar:
                if len(np.unique(y_train_batch)) != len(np.unique(y_test_batch)):
                    continue
                optimizer.zero_grad()
                classifier.fit_from_preprocessed(
                    X_train_batch, y_train_batch, cat_ixs, confs
                )
                predictions = classifier.forward(X_test_batch, return_logits=True)
                loss = loss_function(predictions, y_test_batch.to(config["device"]))
                loss.backward()
                optimizer.step()
                progress_bar.set_postfix(loss=f"{loss.item():.4f}")

        # Evaluate each epoch
        roc_auc, acc, logloss, fairness_df = evaluate_model(
            classifier, eval_config, X_train, y_train, X_test, y_test, edu_test, fam_test
        )
        tag = "Initial" if epoch == 0 else f"Epoch {epoch}"
        print(f"📊 {tag} Utility | ROC: {roc_auc:.4f}, Accuracy: {acc:.4f}, LogLoss: {logloss:.4f}\n")
        print(f"{tag} Fairness Results:\n{fairness_df}\n")

    print("--- ✅ Finetuning Finished ---")


# ============================================================
# 5. Run
# ============================================================
if __name__ == "__main__":
    main()


--- 1. Data Preparation ---
Loaded full data and split: 3500 train, 1500 test samples.
---------------------------

--- 2. Model and Optimizer Setup ---
Using device: cuda
Optimizer: Adam, Finetuning LR: 1e-05
----------------------------------

--- 3. Starting Finetuning & Evaluation ---
📊 Initial Utility | ROC: 0.9992, Accuracy: 0.9933, LogLoss: 0.0216

Initial Fairness Results:
   Sensitive  DP Diff  EO Diff
0  Education   0.1105   0.1044
1     Family   0.0778   0.0833



Finetuning Epoch 1: 100%|██████████| 1/1 [00:01<00:00,  1.46s/it, loss=0.0408]


📊 Epoch 1 Utility | ROC: 0.9994, Accuracy: 0.9927, LogLoss: 0.0207

Epoch 1 Fairness Results:
   Sensitive  DP Diff  EO Diff
0  Education   0.1105   0.1044
1     Family   0.0778   0.0833



Finetuning Epoch 2: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it, loss=0.0410]


📊 Epoch 2 Utility | ROC: 0.9993, Accuracy: 0.9933, LogLoss: 0.0209

Epoch 2 Fairness Results:
   Sensitive  DP Diff  EO Diff
0  Education   0.1089   0.0644
1     Family   0.0778   0.0645



...

📊 Epoch 8 Utility | ROC: 0.9995, Accuracy: 0.9927, LogLoss: 0.0218

Epoch 8 Fairness Results:
   Sensitive  DP Diff  EO Diff
0  Education   0.1082   0.1018
1     Family   0.0778   0.0833



Finetuning Epoch 9: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it, loss=0.0248]


📊 Epoch 9 Utility | ROC: 0.9995, Accuracy: 0.9927, LogLoss: 0.0218

Epoch 9 Fairness Results:
   Sensitive  DP Diff  EO Diff
0  Education   0.1082   0.1018
1     Family   0.0778   0.0833



Finetuning Epoch 10: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it, loss=0.0273]


📊 Epoch 10 Utility | ROC: 0.9996, Accuracy: 0.9927, LogLoss: 0.0217

Epoch 10 Fairness Results:
   Sensitive  DP Diff  EO Diff
0  Education   0.1082   0.1018
1     Family   0.0778   0.0833

--- ✅ Finetuning Finished ---


# heart dataset

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from tabpfn import TabPFNClassifier
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# ====================================================
# --- HEART DATASET ---
# ====================================================

# Load the Heart dataset
heart_df = pd.read_csv('/content/heart.csv')

# Check for missing values
print("Heart Dataset Missing Values:")
print(heart_df.isnull().sum())

# No missing values in heart dataset, so no columns dropped
# Select relevant features
selected_features_heart = ['cp', 'ca', 'thal', 'oldpeak', 'thalach', 'exang', 'age', 'slope']
X_heart = heart_df[selected_features_heart]
y_heart = heart_df['target']

# Sensitive features: sex and age group (binarized)
heart_df['age_group'] = heart_df['age'].apply(lambda x: '>=50' if x >= 50 else '<50')
sensitive_features_heart = {
    'sex': heart_df['sex'],
    'age_group': heart_df['age_group']
}

# Identify numerical & categorical columns
numerical_features_heart = X_heart.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_heart = X_heart.select_dtypes(include=['object']).columns.tolist()

# Preprocessing pipeline
preprocessor_heart = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features_heart),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_features_heart)
    ])

# Apply preprocessing (dense array)
X_heart_processed = preprocessor_heart.fit_transform(X_heart)

# Split data (test_size=0.5 for fairness evaluation)
X_train_heart, X_test_heart, y_train_heart, y_test_heart = train_test_split(
    X_heart_processed, y_heart, test_size=0.5, random_state=42, stratify=y_heart)

# Split sensitive features accordingly
sens_splits_heart = {}
for sens_name, sens in sensitive_features_heart.items():
    _, sens_test, _, _ = train_test_split(sens, y_heart, test_size=0.5, random_state=42, stratify=y_heart)
    sens_splits_heart[sens_name] = sens_test

# Initialize and train classifier
clf_heart = TabPFNClassifier(ignore_pretraining_limits=True)
clf_heart.fit(X_train_heart, y_train_heart)

# Predict
prediction_probabilities_heart = clf_heart.predict_proba(X_test_heart)
predictions_heart = clf_heart.predict(X_test_heart)

# Performance metrics
roc_auc_heart = roc_auc_score(y_test_heart, prediction_probabilities_heart[:, 1])
accuracy_heart = accuracy_score(y_test_heart, predictions_heart)
print("\nHeart Dataset - ROC AUC:", round(roc_auc_heart, 4))
print("Heart Dataset - Accuracy:", round(accuracy_heart, 4))

# Fairness metrics
results_heart = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
for sens_name, sens_test in sens_splits_heart.items():
    dp_diff = fm.demographic_parity_difference(y_test_heart, predictions_heart, sensitive_features=sens_test)
    eo_diff = fm.equalized_odds_difference(y_test_heart, predictions_heart, sensitive_features=sens_test)
    results_heart['Sensitive'].append(sens_name)
    results_heart['DP Diff'].append(round(dp_diff, 4))
    results_heart['EO Diff'].append(round(eo_diff, 4))

# Display results
heart_fairness_df = pd.DataFrame(results_heart)
print("\nHeart Dataset TabPFN Fairness Results:")
print(heart_fairness_df)

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the heart dataset
heart_df = pd.read_csv('/content/heart.csv')

# Prepare features and target
X_heart = heart_df.drop('target', axis=1)
y_heart = heart_df['target']

# Sensitive features: sex and age (binarized)
heart_df['age_group'] = heart_df['age'].apply(lambda x: '>=50' if x >= 50 else '<50')
sensitive_features_heart = {
    'sex': heart_df['sex'],
    'age_group': heart_df['age_group']
}

# Feature selection: Use RandomForest to identify important features
rf_temp = RandomForestClassifier(random_state=42, max_depth=5, min_samples_split=10)
rf_temp.fit(X_heart, y_heart)
feature_importance = pd.Series(rf_temp.feature_importances_, index=X_heart.columns).sort_values(ascending=False)
print("Feature Importance:")
print(feature_importance)

# Select top 8 features to reduce overfitting
top_features = feature_importance.head(8).index.tolist()
X_heart = X_heart[top_features]
print("\nSelected Features:", top_features)

# Identify numerical and categorical columns
numerical_features_heart = X_heart.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_heart = X_heart.select_dtypes(include=['object']).columns.tolist()

# Preprocessing pipeline
preprocessor_heart = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features_heart),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_heart)
    ])

# Split data
X_train_heart, X_test_heart, y_train_heart, y_test_heart = train_test_split(
    X_heart, y_heart, test_size=0.3, random_state=42, stratify=y_heart)

# Train models with regularization
models = {
    'LR': Pipeline([('preprocessor', preprocessor_heart), ('classifier', LogisticRegression(random_state=42, C=0.1))]),
    'RF': Pipeline([('preprocessor', preprocessor_heart), ('classifier', RandomForestClassifier(random_state=42, max_depth=5, min_samples_split=10, n_estimators=50))]),
    'MLP': Pipeline([('preprocessor', preprocessor_heart), ('classifier', MLPClassifier(random_state=42, max_iter=300, hidden_layer_sizes=(50,), alpha=0.01))])
}

# Dictionary to store results
results_heart = {'Model': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': [], 'Cross-Val Mean': [], 'Cross-Val Std': []}

for name, model in models.items():
    # Fit and predict
    model.fit(X_train_heart, y_train_heart)
    y_pred_heart = model.predict(X_test_heart)

    # Test set accuracy
    acc = accuracy_score(y_test_heart, y_pred_heart)

    # Cross-validation score
    cv_scores = cross_val_score(model, X_heart, y_heart, cv=5, scoring='accuracy')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    # Fairness metrics for each sensitive feature
    for sens_name, sens in sensitive_features_heart.items():
        sens_train, sens_test = train_test_split(sens, test_size=0.3, random_state=42, stratify=y_heart)
        dp_diff = fm.demographic_parity_difference(y_test_heart, y_pred_heart, sensitive_features=sens_test)
        eo_diff = fm.equalized_odds_difference(y_test_heart, y_pred_heart, sensitive_features=sens_test)

        results_heart['Model'].append(name)
        results_heart['Sensitive'].append(sens_name)
        results_heart['DP Diff'].append(round(dp_diff, 4))
        results_heart['EO Diff'].append(round(eo_diff, 4))
        results_heart['Accuracy'].append(round(acc, 4))
        results_heart['Cross-Val Mean'].append(round(cv_mean, 4))
        results_heart['Cross-Val Std'].append(round(cv_std, 4))

# Display results as DataFrame
heart_results_df = pd.DataFrame(results_heart)
print("\nHeart Dataset Fairness Results (Optimized):")
print(heart_results_df)

Feature Importance:
cp          0.180657
ca          0.134886
thal        0.130811
oldpeak     0.129611
thalach     0.106730
exang       0.077248
age         0.065533
slope       0.049775
trestbps    0.044354
chol        0.034428
sex         0.032386
restecg     0.009131
fbs         0.004451
dtype: float64

Selected Features: ['cp', 'ca', 'thal', 'oldpeak', 'thalach', 'exang', 'age', 'slope']

Heart Dataset Fairness Results (Optimized):
  Model  Sensitive  DP Diff  EO Diff  Accuracy  Cross-Val Mean  Cross-Val Std
0    LR        sex   0.2279   0.2000    0.8409          0.8410         0.0303
1    LR  age_group   0.2018   0.1233    0.8409          0.8410         0.0303
2    RF        sex   0.3149   0.1001    0.9123          0.9024         0.0269
3    RF  age_group   0.1668   0.0738    0.9123          0.9024         0.0269
4   MLP        sex   0.2815   0.0401    0.8994          0.8888         0.0238
5   MLP  age_group   0.2231   0.1530    0.8994          0.8888         0.0238


In [ ]:
"""
Full fine-tuning & fairness evaluation for HEART dataset
using TabPFNClassifier (finetune version).
"""

import pandas as pd
import numpy as np
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
import fairlearn.metrics as fm
from tabpfn import TabPFNClassifier
from tabpfn.utils import meta_dataset_collator
from tabpfn.finetune_utils import clone_model_for_evaluation
import warnings
warnings.filterwarnings("ignore")

# ========== COMMON HELPERS ==========

def to_numpy_safe(x):
    """Convert Tensor/list to numpy safely."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    elif isinstance(x, (list, tuple)):
        return np.array(x)
    else:
        return x

def preprocess_dataset(df, target_col, drop_cols=None):
    """Preprocess generic tabular dataset (standardize + onehot)."""
    if drop_cols:
        df = df.drop(columns=drop_cols, errors='ignore')
    X = df.drop(columns=[target_col])
    y = df[target_col].values

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), cat_cols)
    ])
    X_proc = preprocessor.fit_transform(X)
    return X_proc, y, X, df

def setup_tabpfn_and_optimizer(config):
    """Initialize TabPFN model + optimizer."""
    classifier_cfg = {
        "ignore_pretraining_limits": True,
        "device": config["device"],
        "n_estimators": 2,
        "random_state": config["random_seed"],
        "inference_precision": torch.float32,
    }
    clf = TabPFNClassifier(**classifier_cfg, fit_mode="batched", differentiable_input=False)
    clf._initialize_model_variables()
    optimizer = Adam(clf.model_.parameters(), lr=config["finetuning"]["learning_rate"])
    return clf, optimizer, classifier_cfg

def evaluate_fairness_and_performance(clf, eval_cfg, X_train, y_train, X_test, y_test, sensitive_dict):
    """Evaluate ROC/AUC/ACC + fairness (DP diff, EO diff)."""
    eval_clf = clone_model_for_evaluation(clf, eval_cfg, TabPFNClassifier)
    eval_clf.fit(X_train, y_train)
    probs = eval_clf.predict_proba(X_test)
    preds = (probs[:, 1] > 0.5).astype(int)
    roc_auc = roc_auc_score(y_test, probs[:, 1])
    acc = accuracy_score(y_test, preds)
    ll = log_loss(y_test, probs)

    fairness = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
    for s_name, s_feat in sensitive_dict.items():
        dp = fm.demographic_parity_difference(y_test, preds, sensitive_features=s_feat)
        eo = fm.equalized_odds_difference(y_test, preds, sensitive_features=s_feat)
        fairness['Sensitive'].append(s_name)
        fairness['DP Diff'].append(round(dp, 4))
        fairness['EO Diff'].append(round(eo, 4))
    fairness_df = pd.DataFrame(fairness)
    return roc_auc, acc, ll, fairness_df

def finetune_and_evaluate(dataset_name, df, target, sensitive_feats, drop_cols, config):
    """Full workflow for one dataset."""
    print(f"\n##### WORKFLOW FOR {dataset_name.upper()} #####")

    # Preprocess
    print(f"=== Preparing {dataset_name.upper()} dataset ===")
    print(f"{dataset_name} missing values:\n", df.isnull().sum())
    X_proc, y, X_raw, df_raw = preprocess_dataset(df, target, drop_cols)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_proc, y, test_size=0.3, random_state=config["random_seed"], stratify=y
    )
    sens_splits = {}
    for s_name, s_series in sensitive_feats.items():
        _, s_test, _, _ = train_test_split(
            s_series, y, test_size=0.3, random_state=config["random_seed"], stratify=y
        )
        sens_splits[s_name] = s_test

    print(f"{dataset_name}: {X_train.shape[0]} train, {X_test.shape[0]} test")

    # Setup model & optimizer
    print("=== Setting up TabPFN model & optimizer ===")
    clf, optimizer, clf_cfg = setup_tabpfn_and_optimizer(config)
    print(f"Device: {config['device']}, finetune LR: {config['finetuning']['learning_rate']}")

    # Prepare dataloader
    train_datasets = clf.get_preprocessed_datasets(
        X_train, y_train, train_test_split, config["finetuning"]["batch_size"]
    )
    dataloader = DataLoader(train_datasets,
                            batch_size=config["finetuning"]["meta_batch_size"],
                            collate_fn=meta_dataset_collator)
    loss_fn = torch.nn.CrossEntropyLoss()
    eval_cfg = {**clf_cfg, "inference_config": {"SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]}}

    print("Starting finetuning & evaluation loop...\n")

    # Finetune loop
    for epoch in range(config["finetuning"]["epochs"] + 1):
        if epoch > 0:
            progress = tqdm(dataloader, desc=f"{dataset_name.upper()} Finetune Epoch {epoch}")
            for (X_tr_b, X_te_b, y_tr_b, y_te_b, cat_ixs, confs) in progress:
                y_tr_np, y_te_np = to_numpy_safe(y_tr_b), to_numpy_safe(y_te_b)
                if len(np.unique(y_tr_np)) != len(np.unique(y_te_np)):
                    continue
                optimizer.zero_grad()
                clf.fit_from_preprocessed(X_tr_b, y_tr_b, cat_ixs, confs)
                logits = clf.forward(X_te_b, return_logits=True)
                loss = loss_fn(logits, y_te_b.to(config["device"]))
                loss.backward()
                optimizer.step()
                progress.set_postfix(loss=f"{loss.item():.4f}")

        # Evaluate
        roc_auc, acc, ll, fairness = evaluate_fairness_and_performance(
            clf, eval_cfg, X_train, y_train, X_test, y_test, sens_splits
        )
        phase = "Initial" if epoch == 0 else f"Epoch {epoch}"
        print(f"\n{dataset_name.upper()} | {phase} -> ROC AUC: {roc_auc:.4f}, Acc: {acc:.4f}, LogLoss: {ll:.4f}")
        print(f"{dataset_name.upper()} Fairness:\n{fairness}\n")

    print(f"##### {dataset_name.upper()} FINISHED #####\n")


# ========== MAIN ENTRY ==========

def main():
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "n_inference_context_samples": 5000,
        "finetuning": {
            "epochs": 10,                # try small for testing
            "learning_rate": 1e-5,
            "meta_batch_size": 1,
            "batch_size": 5000
        }
    }

    # --- HEART DATASET ---
    heart_df = pd.read_csv("/content/heart.csv")
    heart_df["age_group"] = heart_df["age"].apply(lambda x: ">=50" if x >= 50 else "<50")
    sens_heart = {
        "sex": heart_df["sex"],
        "age_group": heart_df["age_group"]
    }
    finetune_and_evaluate(
        "heart",
        heart_df,
        target="target",
        sensitive_feats=sens_heart,
        drop_cols=None,
        config=config
    )


if __name__ == "__main__":
    main()

# adult dataset

In [ ]:
"""
Full dataset fine-tuning for Adult dataset with fairness evaluation.
No subsampling; uses entire dataset.
"""
from functools import partial
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from torch.optim import Adam, Optimizer
from torch.utils.data import DataLoader
from tqdm import tqdm
from tabpfn import TabPFNClassifier
from tabpfn.finetune_utils import clone_model_for_evaluation
from tabpfn.utils import meta_dataset_collator
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

def prepare_data(config: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Loads and splits the full Adult dataset, including sensitive features."""
    print("--- 1. Data Preparation ---")
    adult_df = pd.read_csv('/content/adult.csv')
    X_all = adult_df.drop('income>50K', axis=1)
    y_all = adult_df['income>50K']
    race_all = adult_df['race']
    sex_all = adult_df['sex']

    # Preprocessing: Standardize numerical, one-hot categorical
    numerical_features = X_all.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X_all.select_dtypes(include=['object']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
        ])
    X_all_processed = preprocessor.fit_transform(X_all)

    # No subsampling: use full data
    X_all = X_all_processed
    y = y_all.values
    race = race_all.values
    sex = sex_all.values

    splitter = partial(
        train_test_split,
        test_size=config["valid_set_ratio"],
        random_state=config["random_seed"],
    )
    X_train, X_test, y_train, y_test = splitter(X_all, y, stratify=y)
    _, race_test, _, _ = splitter(race, y, stratify=y)
    _, sex_test, _, _ = splitter(sex, y, stratify=y)

    print(
        f"Loaded full data and split: {X_train.shape[0]} train, {X_test.shape[0]} test samples."
    )
    print("---------------------------\n")
    return X_train, X_test, y_train, y_test, race_test, sex_test

def setup_model_and_optimizer(config: dict) -> tuple[TabPFNClassifier, Optimizer, dict]:
    """Initializes the TabPFN classifier, optimizer, and training configs."""
    print("--- 2. Model and Optimizer Setup ---")
    classifier_config = {
        "ignore_pretraining_limits": True,
        "device": config["device"],
        "n_estimators": 2,
        "random_state": config["random_seed"],
        "inference_precision": torch.float32,
    }
    classifier = TabPFNClassifier(
        **classifier_config, fit_mode="batched", differentiable_input=False
    )
    classifier._initialize_model_variables()
    # Optimizer uses finetuning-specific learning rate
    optimizer = Adam(
        classifier.model_.parameters(), lr=config["finetuning"]["learning_rate"]
    )
    print(f"Using device: {config['device']}")
    print(f"Optimizer: Adam, Finetuning LR: {config['finetuning']['learning_rate']}")
    print("----------------------------------\n")
    return classifier, optimizer, classifier_config

def evaluate_model(
    classifier: TabPFNClassifier,
    eval_config: dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    race_test: np.ndarray,
    sex_test: np.ndarray,
) -> tuple[float, float, float, pd.DataFrame]:
    """Evaluates utility and fairness on the test set."""
    eval_classifier = clone_model_for_evaluation(
        classifier, eval_config, TabPFNClassifier
    )
    eval_classifier.fit(X_train, y_train)
    try:
        probabilities = eval_classifier.predict_proba(X_test)
        predictions = (probabilities[:, 1] > 0.5).astype(int)
        roc_auc = roc_auc_score(y_test, probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)
        log_loss_score = log_loss(y_test, probabilities)

        # Fairness metrics
        sensitive = {'race': race_test, 'sex': sex_test}
        results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
        for sens_name, sens_test in sensitive.items():
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
        fairness_df = pd.DataFrame(results)
    except Exception as e:
        print(f"An error occurred during evaluation: {e}")
        roc_auc, accuracy, log_loss_score = np.nan, np.nan, np.nan
        fairness_df = pd.DataFrame()
    return roc_auc, accuracy, log_loss_score, fairness_df

def main() -> None:
    """Main function to configure and run the finetuning workflow with fairness eval."""
    # --- Master Configuration ---
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "valid_set_ratio": 0.3,
        "n_inference_context_samples": 5000,  # Reduced for full data to save memory
    }
    config["finetuning"] = {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 5000,  # Adjusted for full data
    }
    # --- Setup Data, Model, and Dataloader ---
    X_train, X_test, y_train, y_test, race_test, sex_test = prepare_data(config)
    classifier, optimizer, classifier_config = setup_model_and_optimizer(config)
    splitter = partial(train_test_split, test_size=config["valid_set_ratio"])
    training_datasets = classifier.get_preprocessed_datasets(
        X_train, y_train, splitter, config["finetuning"]["batch_size"]
    )
    finetuning_dataloader = DataLoader(
        training_datasets,
        batch_size=config["finetuning"]["meta_batch_size"],
        collate_fn=meta_dataset_collator,
    )
    loss_function = torch.nn.CrossEntropyLoss()
    eval_config = {
        **classifier_config,
        "inference_config": {
            "SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]
        },
    }
    # --- Finetuning and Evaluation Loop ---
    print("--- 3. Starting Finetuning & Evaluation ---")
    for epoch in range(config["finetuning"]["epochs"] + 1):
        if epoch > 0:
            # Finetuning Step
            progress_bar = tqdm(finetuning_dataloader, desc=f"Finetuning Epoch {epoch}")
            for (
                X_train_batch,
                X_test_batch,
                y_train_batch,
                y_test_batch,
                cat_ixs,
                confs,
            ) in progress_bar:
                if len(np.unique(y_train_batch)) != len(np.unique(y_test_batch)):
                    continue
                optimizer.zero_grad()
                classifier.fit_from_preprocessed(
                    X_train_batch, y_train_batch, cat_ixs, confs
                )
                predictions = classifier.forward(X_test_batch, return_logits=True)
                loss = loss_function(predictions, y_test_batch.to(config["device"]))
                loss.backward()
                optimizer.step()
                progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        # Evaluation Step
        epoch_roc, epoch_acc, epoch_log_loss, fairness_df = evaluate_model(
            classifier, eval_config, X_train, y_train, X_test, y_test, race_test, sex_test
        )
        status = "Initial" if epoch == 0 else f"Epoch {epoch}"
        print(
            f"📊 {status} Utility | Test ROC: {epoch_roc:.4f}, Accuracy: {epoch_acc:.4f}, Log Loss: {epoch_log_loss:.4f}\n"
        )
        print(f"{status} Fairness Results:\n{fairness_df}\n")
    print("--- ✅ Finetuning Finished ---")

if __name__ == "__main__":
    main()

--- 1. Data Preparation ---
Loaded full data and split: 34189 train, 14653 test samples.
---------------------------

--- 2. Model and Optimizer Setup ---
Using device: cuda
Optimizer: Adam, Finetuning LR: 1e-05
----------------------------------

--- 3. Starting Finetuning & Evaluation ---
📊 Initial Utility | Test ROC: 0.9151, Accuracy: 0.8606, Log Loss: 0.3018

Initial Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1581   0.2592
1       sex   0.1818   0.1325



Finetuning Epoch 1: 100%|██████████| 7/7 [00:15<00:00,  2.17s/it, loss=0.3215]


📊 Epoch 1 Utility | Test ROC: 0.9158, Accuracy: 0.8617, Log Loss: 0.3007

Epoch 1 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1530   0.1708
1       sex   0.1799   0.1225



Finetuning Epoch 2: 100%|██████████| 7/7 [00:15<00:00,  2.14s/it, loss=0.3077]


📊 Epoch 2 Utility | Test ROC: 0.9158, Accuracy: 0.8617, Log Loss: 0.2999

Epoch 2 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1494   0.1701
1       sex   0.1837   0.1352



...

📊 Epoch 8 Utility | Test ROC: 0.9205, Accuracy: 0.8654, Log Loss: 0.2908

Epoch 8 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1558   0.1717
1       sex   0.1814   0.1401



Finetuning Epoch 9: 100%|██████████| 7/7 [00:15<00:00,  2.14s/it, loss=0.3151]


📊 Epoch 9 Utility | Test ROC: 0.9205, Accuracy: 0.8645, Log Loss: 0.2906

Epoch 9 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1726   0.2642
1       sex   0.1828   0.1458



Finetuning Epoch 10: 100%|██████████| 7/7 [00:15<00:00,  2.14s/it, loss=0.2862]


📊 Epoch 10 Utility | Test ROC: 0.9209, Accuracy: 0.8649, Log Loss: 0.2903

Epoch 10 Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1581   0.1736
1       sex   0.1815   0.1396

--- ✅ Finetuning Finished ---


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the adult dataset
adult_df = pd.read_csv('/content/adult.csv')

# Prepare features and target
X = adult_df.drop('income>50K', axis=1)
y = adult_df['income>50K']

# Sensitive features: race and sex (assuming they are encoded as integers)
sensitive_features = {
    'race': adult_df['race'],
    'sex': adult_df['sex']
}

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()  # If any

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train models
models = {
    'LR': Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(random_state=42))]),
    'RF': Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor), ('classifier', MLPClassifier(random_state=42, max_iter=300))])
}

# Dictionary to store results
results = {'Model': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': []}

for name, model in models.items():
    # Fit and predict
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    # Fairness metrics for each sensitive feature
    for sens_name, sens in sensitive_features.items():
        sens_train, sens_test = train_test_split(sens, test_size=0.2, random_state=42, stratify=y)
        dp_diff = fm.demographic_parity_difference(y_test, y_pred, sensitive_features=sens_test)
        eo_diff = fm.equalized_odds_difference(y_test, y_pred, sensitive_features=sens_test)

        results['Model'].append(name)
        results['Sensitive'].append(sens_name)
        results['DP Diff'].append(round(dp_diff, 4))
        results['EO Diff'].append(round(eo_diff, 4))
        results['Accuracy'].append(round(acc, 4))

# Display results as DataFrame
adult_results_df = pd.DataFrame(results)
print("Adult Dataset Fairness Results:")
print(adult_results_df)

Adult Dataset Fairness Results:
  Model Sensitive  DP Diff  EO Diff  Accuracy
0    LR      race   0.1821   0.2646    0.8427
1    LR       sex   0.1888   0.1892    0.8427
2    RF      race   0.1433   0.3030    0.8561
3    RF       sex   0.1763   0.0845    0.8561
4   MLP      race   0.1837   0.2634    0.8564
5   MLP       sex   0.1960   0.1535    0.8564


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from tabpfn import TabPFNClassifier
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the adult dataset
adult_df = pd.read_csv('/content/adult.csv')

# Prepare features and target
X = adult_df.drop('income>50K', axis=1)
y = adult_df['income>50K']

# Sensitive features: race and sex
sensitive_features = {
    'race': adult_df['race'],
    'sex': adult_df['sex']
}

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()  # If any

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ])

# Apply preprocessing
X_processed = preprocessor.fit_transform(X)

# Split data (test_size=0.5)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.5, random_state=42, stratify=y)

# Split sensitive features accordingly
sens_splits = {}
for sens_name, sens in sensitive_features.items():
    _, sens_test, _, _ = train_test_split(sens, y, test_size=0.5, random_state=42, stratify=y)
    sens_splits[sens_name] = sens_test

# Initialize and fit the classifier
clf = TabPFNClassifier(ignore_pretraining_limits=True)
clf.fit(X_train, y_train)

# Predict probabilities and labels
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

# Calculate performance metrics
roc_auc = roc_auc_score(y_test, prediction_probabilities[:, 1])
accuracy = accuracy_score(y_test, predictions)

print("Adult Dataset - ROC AUC:", round(roc_auc, 4))
print("Adult Dataset - Accuracy:", round(accuracy, 4))

# Fairness metrics
results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
for sens_name, sens_test in sens_splits.items():
    dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
    eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)

    results['Sensitive'].append(sens_name)
    results['DP Diff'].append(round(dp_diff, 4))
    results['EO Diff'].append(round(eo_diff, 4))

# Display fairness results
adult_fairness_df = pd.DataFrame(results)
print("\nAdult Dataset TabPFN Fairness Results:")
print(adult_fairness_df)

tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

Adult Dataset - ROC AUC: 0.9193
Adult Dataset - Accuracy: 0.8647

Adult Dataset TabPFN Fairness Results:
  Sensitive  DP Diff  EO Diff
0      race   0.1788   0.2513
1       sex   0.1790   0.0896


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from tabpfn import TabPFNClassifier
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the adult dataset
adult_df = pd.read_csv('/content/adult.csv')

# Prepare features and target
X = adult_df.drop('income>50K', axis=1)
y = adult_df['income>50K']

# Sensitive features: race and sex
sensitive_features = {
    'race': adult_df['race'],
    'sex': adult_df['sex']
}

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ])

# Define dataset sizes to test
dataset_sizes = [10000, 1000, 500]

# Dictionary to store results
results = {'Dataset Size': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': [], 'ROC AUC': []}

# Run experiments for each dataset size
for size in dataset_sizes:
    # Stratified sampling to create subset
    sss = StratifiedShuffleSplit(n_splits=1, train_size=size, random_state=42)
    for train_index, _ in sss.split(X, y):
        X_subset = X.iloc[train_index]
        y_subset = y.iloc[train_index]
        sens_subset_race = sensitive_features['race'].iloc[train_index]
        sens_subset_sex = sensitive_features['sex'].iloc[train_index]

        # Apply preprocessing
        X_subset_processed = preprocessor.fit_transform(X_subset)

        # Split subset into train and test
        X_train, X_test, y_train, y_test, sens_train_race, sens_test_race, sens_train_sex, sens_test_sex = train_test_split(
            X_subset_processed, y_subset, sens_subset_race, sens_subset_sex,
            test_size=0.5, random_state=42, stratify=y_subset
        )

        # Initialize and fit the classifier
        clf = TabPFNClassifier(ignore_pretraining_limits=True)
        clf.fit(X_train, y_train)

        # Predict probabilities and labels
        prediction_probabilities = clf.predict_proba(X_test)
        predictions = clf.predict(X_test)

        # Calculate performance metrics
        roc_auc = roc_auc_score(y_test, prediction_probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)

        # Fairness metrics
        for sens_name, sens_test in [('race', sens_test_race), ('sex', sens_test_sex)]:
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)

            # Store results
            results['Dataset Size'].append(size)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
            results['Accuracy'].append(round(accuracy, 4))
            results['ROC AUC'].append(round(roc_auc, 4))

# Display results as DataFrame
results_df = pd.DataFrame(results)
print("Adult Dataset TabPFN Fairness Results for Different Sizes:")
print(results_df)

tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

Adult Dataset TabPFN Fairness Results for Different Sizes:
   Dataset Size Sensitive  DP Diff  EO Diff  Accuracy  ROC AUC
0         10000      race   0.2417   0.7027    0.8634   0.9147
1         10000       sex   0.1690   0.0611    0.8634   0.9147
2          1000      race   0.2000   0.5534    0.8560   0.8957
3          1000       sex   0.1535   0.0529    0.8560   0.8957
4           500      race   0.3551   0.5577    0.8320   0.8845
5           500       sex   0.2110   0.5273    0.8320   0.8845


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

# Load the adult dataset
adult_df = pd.read_csv('/content/adult.csv')

# Prepare features and target
X = adult_df.drop('income>50K', axis=1)
y = adult_df['income>50K']

# Sensitive features: race and sex
sensitive_features = {
    'race': adult_df['race'],
    'sex': adult_df['sex']
}

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing pipeline for non-TabPFN models
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Define dataset sizes to test
dataset_sizes = [10000, 1000, 500]

# Models to evaluate
models = {
    'LR': Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(random_state=42))]),
    'RF': Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor), ('classifier', MLPClassifier(random_state=42, max_iter=300))]),
}

# Dictionary to store results
results = {'Dataset Size': [], 'Model': [], 'Sensitive': [], 'DP Diff': [], 'EO Diff': [], 'Accuracy': []}

# Run experiments for each dataset size
for size in dataset_sizes:
    # Stratified sampling to create subset
    sss = StratifiedShuffleSplit(n_splits=1, train_size=size, random_state=42)
    for train_index, _ in sss.split(X, y):
        X_subset = X.iloc[train_index]
        y_subset = y.iloc[train_index]
        sens_subset_race = sensitive_features['race'].iloc[train_index]
        sens_subset_sex = sensitive_features['sex'].iloc[train_index]

        # Split subset into train and test
        X_train, X_test, y_train, y_test, sens_train_race, sens_test_race, sens_train_sex, sens_test_sex = train_test_split(
            X_subset, y_subset, sens_subset_race, sens_subset_sex,
            test_size=0.2, random_state=42, stratify=y_subset
        )

        # Evaluate each model
        for name, model in models.items():
            # Fit and predict
            if name == 'TabPFN':
                # TabPFN handles preprocessing internally
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
            else:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

            # Calculate accuracy
            acc = accuracy_score(y_test, y_pred)

            # Fairness metrics for each sensitive feature
            for sens_name, sens_test in [('race', sens_test_race), ('sex', sens_test_sex)]:
                dp_diff = fm.demographic_parity_difference(y_test, y_pred, sensitive_features=sens_test)
                eo_diff = fm.equalized_odds_difference(y_test, y_pred, sensitive_features=sens_test)

                # Store results
                results['Dataset Size'].append(size)
                results['Model'].append(name)
                results['Sensitive'].append(sens_name)
                results['DP Diff'].append(round(dp_diff, 4))
                results['EO Diff'].append(round(eo_diff, 4))
                results['Accuracy'].append(round(acc, 4))

# Display results as DataFrame
results_df = pd.DataFrame(results)
print("Adult Dataset Fairness Results for Different Sizes:")
print(results_df)

Adult Dataset Fairness Results for Different Sizes:
    Dataset Size Model Sensitive  DP Diff  EO Diff  Accuracy
0          10000    LR      race   0.2241   0.5804    0.8430
1          10000    LR       sex   0.2142   0.2350    0.8430
2          10000    RF      race   0.1914   0.6410    0.8565
3          10000    RF       sex   0.2002   0.1118    0.8565
4          10000   MLP      race   0.2086   0.6247    0.8430
5          10000   MLP       sex   0.2053   0.1138    0.8430
6           1000    LR      race   0.1503   0.5000    0.8450
7           1000    LR       sex   0.1917   0.2906    0.8450
8           1000    RF      race   0.1734   0.5116    0.8300
9           1000    RF       sex   0.1938   0.3162    0.8300
10          1000   MLP      race   0.1734   0.5581    0.8550
11          1000   MLP       sex   0.2057   0.3675    0.8550
12           500    LR      race   0.4000   0.6667    0.8300
13           500    LR       sex   0.1757   0.4545    0.8300
14           500    RF      race 

In [ ]:
"""
Finetuning for Adult dataset with fairness evaluation across different dataset sizes.
Subsamples data at specified levels (10000, 1000, 500) instead of using full dataset.
"""
from functools import partial
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from torch.optim import Adam, Optimizer
from torch.utils.data import DataLoader
from tqdm import tqdm
from tabpfn import TabPFNClassifier
from tabpfn.finetune_utils import clone_model_for_evaluation
from tabpfn.utils import meta_dataset_collator
import fairlearn.metrics as fm
import warnings
warnings.filterwarnings('ignore')

def prepare_data(config: dict, dataset_size: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Loads and splits the Adult dataset with subsampling."""
    print(f"--- 1. Data Preparation (Size: {dataset_size}) ---")
    adult_df = pd.read_csv('/content/adult.csv')
    X_all = adult_df.drop('income>50K', axis=1)
    y_all = adult_df['income>50K']
    race_all = adult_df['race']
    sex_all = adult_df['sex']

    # Subsample data
    sss = StratifiedShuffleSplit(n_splits=1, train_size=min(dataset_size, len(X_all)), random_state=config["random_seed"])
    for train_index, _ in sss.split(X_all, y_all):
        X_subset = X_all.iloc[train_index]
        y_subset = y_all.iloc[train_index]
        race_subset = race_all.iloc[train_index]
        sex_subset = sex_all.iloc[train_index]

    # Preprocessing: Standardize numerical, one-hot categorical
    numerical_features = X_subset.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X_subset.select_dtypes(include=['object']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
        ])
    X_subset_processed = preprocessor.fit_transform(X_subset)

    # Split subset
    splitter = partial(
        train_test_split,
        test_size=config["valid_set_ratio"],
        random_state=config["random_seed"],
    )
    X_train, X_test, y_train, y_test = splitter(X_subset_processed, y_subset, stratify=y_subset)
    _, race_test, _, _ = splitter(race_subset, y_subset, stratify=y_subset)
    _, sex_test, _, _ = splitter(sex_subset, y_subset, stratify=y_subset)

    print(
        f"Loaded subset (size: {dataset_size}) and split: {X_train.shape[0]} train, {X_test.shape[0]} test samples."
    )
    print("---------------------------\n")
    return X_train, X_test, y_train, y_test, race_test, sex_test

def setup_model_and_optimizer(config: dict) -> tuple[TabPFNClassifier, Optimizer, dict]:
    """Initializes the TabPFN classifier, optimizer, and training configs."""
    print("--- 2. Model and Optimizer Setup ---")
    classifier_config = {
        "ignore_pretraining_limits": True,
        "device": config["device"],
        "n_estimators": 2,
        "random_state": config["random_seed"],
        "inference_precision": torch.float32,
    }
    classifier = TabPFNClassifier(
        **classifier_config, fit_mode="batched", differentiable_input=False
    )
    classifier._initialize_model_variables()
    # Optimizer uses finetuning-specific learning rate
    optimizer = Adam(
        classifier.model_.parameters(), lr=config["finetuning"]["learning_rate"]
    )
    print(f"Using device: {config['device']}")
    print(f"Optimizer: Adam, Finetuning LR: {config['finetuning']['learning_rate']}")
    print("----------------------------------\n")
    return classifier, optimizer, classifier_config

def evaluate_model(
    classifier: TabPFNClassifier,
    eval_config: dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    race_test: np.ndarray,
    sex_test: np.ndarray,
) -> tuple[float, float, float, pd.DataFrame]:
    """Evaluates utility and fairness on the test set."""
    eval_classifier = clone_model_for_evaluation(
        classifier, eval_config, TabPFNClassifier
    )
    eval_classifier.fit(X_train, y_train)
    try:
        probabilities = eval_classifier.predict_proba(X_test)
        predictions = (probabilities[:, 1] > 0.5).astype(int)
        roc_auc = roc_auc_score(y_test, probabilities[:, 1])
        accuracy = accuracy_score(y_test, predictions)
        log_loss_score = log_loss(y_test, probabilities)
        # Fairness metrics
        sensitive = {'race': race_test, 'sex': sex_test}
        results = {'Sensitive': [], 'DP Diff': [], 'EO Diff': []}
        for sens_name, sens_test in sensitive.items():
            dp_diff = fm.demographic_parity_difference(y_test, predictions, sensitive_features=sens_test)
            eo_diff = fm.equalized_odds_difference(y_test, predictions, sensitive_features=sens_test)
            results['Sensitive'].append(sens_name)
            results['DP Diff'].append(round(dp_diff, 4))
            results['EO Diff'].append(round(eo_diff, 4))
        fairness_df = pd.DataFrame(results)
    except Exception as e:
        print(f"An error occurred during evaluation: {e}")
        roc_auc, accuracy, log_loss_score = np.nan, np.nan, np.nan
        fairness_df = pd.DataFrame()
    return roc_auc, accuracy, log_loss_score, fairness_df

def main() -> None:
    """Main function to configure and run the finetuning workflow with fairness eval for different dataset sizes."""
    # --- Master Configuration ---
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "valid_set_ratio": 0.3,
        "n_inference_context_samples": 5000,  # Reduced for full data to save memory
    }
    config["finetuning"] = {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 5000,  # Adjusted for subsampled data
    }

    # Define dataset sizes to test
    dataset_sizes = [10000, 1000, 500]

    # Loop over dataset sizes
    for dataset_size in dataset_sizes:
        print(f"\n=== Processing Dataset Size: {dataset_size} ===\n")
        # --- Setup Data, Model, and Dataloader ---
        X_train, X_test, y_train, y_test, race_test, sex_test = prepare_data(config, dataset_size)
        classifier, optimizer, classifier_config = setup_model_and_optimizer(config)
        splitter = partial(train_test_split, test_size=config["valid_set_ratio"])
        training_datasets = classifier.get_preprocessed_datasets(
            X_train, y_train, splitter, config["finetuning"]["batch_size"]
        )
        finetuning_dataloader = DataLoader(
            training_datasets,
            batch_size=config["finetuning"]["meta_batch_size"],
            collate_fn=meta_dataset_collator,
        )
        loss_function = torch.nn.CrossEntropyLoss()
        eval_config = {
            **classifier_config,
            "inference_config": {
                "SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]
            },
        }

        # --- Finetuning and Evaluation Loop ---
        print("--- 3. Starting Finetuning & Evaluation ---")
        for epoch in range(config["finetuning"]["epochs"] + 1):
            if epoch > 0:
                # Finetuning Step
                progress_bar = tqdm(finetuning_dataloader, desc=f"Finetuning Epoch {epoch} (Size: {dataset_size})")
                for (
                    X_train_batch,
                    X_test_batch,
                    y_train_batch,
                    y_test_batch,
                    cat_ixs,
                    confs,
                ) in progress_bar:
                    if len(np.unique(y_train_batch)) != len(np.unique(y_test_batch)):
                        continue
                    optimizer.zero_grad()
                    classifier.fit_from_preprocessed(
                        X_train_batch, y_train_batch, cat_ixs, confs
                    )
                    predictions = classifier.forward(X_test_batch, return_logits=True)
                    loss = loss_function(predictions, y_test_batch.to(config["device"]))
                    loss.backward()
                    optimizer.step()
                    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

            # Evaluation Step
            epoch_roc, epoch_acc, epoch_log_loss, fairness_df = evaluate_model(
                classifier, eval_config, X_train, y_train, X_test, y_test, race_test, sex_test
            )
            status = "Initial" if epoch == 0 else f"Epoch {epoch}"
            print(
                f"📊 {status} Utility (Size: {dataset_size}) | Test ROC: {epoch_roc:.4f}, Accuracy: {epoch_acc:.4f}, Log Loss: {epoch_log_loss:.4f}\n"
            )
            print(f"{status} Fairness Results (Size: {dataset_size}):\n{fairness_df}\n")

        print(f"--- ✅ Finetuning Finished for Size {dataset_size} ---")

if __name__ == "__main__":
    main()


=== Processing Dataset Size: 10000 ===

--- 1. Data Preparation (Size: 10000) ---
Loaded subset (size: 10000) and split: 7000 train, 3000 test samples.
---------------------------

--- 2. Model and Optimizer Setup ---
Using device: cuda
Optimizer: Adam, Finetuning LR: 1e-05
----------------------------------

--- 3. Starting Finetuning & Evaluation ---
📊 Initial Utility (Size: 10000) | Test ROC: 0.9118, Accuracy: 0.8537, Log Loss: 0.3103

Initial Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.1560   0.6364
1       sex   0.1869   0.0727



Finetuning Epoch 1 (Size: 10000): 100%|██████████| 2/2 [00:02<00:00,  1.48s/it, loss=0.3252]


📊 Epoch 1 Utility (Size: 10000) | Test ROC: 0.9117, Accuracy: 0.8543, Log Loss: 0.3095

Epoch 1 Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.1560   0.6364
1       sex   0.1845   0.0709



Finetuning Epoch 2 (Size: 10000): 100%|██████████| 2/2 [00:02<00:00,  1.31s/it, loss=0.2912]


📊 Epoch 2 Utility (Size: 10000) | Test ROC: 0.9119, Accuracy: 0.8550, Log Loss: 0.3090

Epoch 2 Fairness Results (Size: 10000):
  Sensitive  DP Diff  EO Diff
0      race   0.1560   0.6364
1       sex   0.1825   0.0688



...

📊 Epoch 8 Utility (Size: 500) | Test ROC: 0.8816, Accuracy: 0.8400, Log Loss: 0.3546

Epoch 8 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.2981   0.7500
1       sex   0.2104   0.5938



Finetuning Epoch 9 (Size: 500): 100%|██████████| 1/1 [00:00<00:00,  3.15it/s, loss=0.3937]


📊 Epoch 9 Utility (Size: 500) | Test ROC: 0.8816, Accuracy: 0.8400, Log Loss: 0.3543

Epoch 9 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.2981   0.7500
1       sex   0.2104   0.5938



Finetuning Epoch 10 (Size: 500): 100%|██████████| 1/1 [00:00<00:00,  3.25it/s, loss=0.3437]


📊 Epoch 10 Utility (Size: 500) | Test ROC: 0.8813, Accuracy: 0.8400, Log Loss: 0.3542

Epoch 10 Fairness Results (Size: 500):
  Sensitive  DP Diff  EO Diff
0      race   0.2981   0.7500
1       sex   0.2104   0.5938

--- ✅ Finetuning Finished for Size 500 ---
